# AutoAWT - Automatic Atrial Wall Thickness
## Google Colab Notebook (GPU Accelerated)

This notebook runs the AutoAWT pipeline for measuring 3D atrial wall thickness from cardiac CT data.

**Input formats supported:**
- **BMP masks** + DICOM metadata (original pipeline — pre-segmented masks)
- **DICOM series** (direct import — reads pixel data, auto-threshold)
- **NIfTI** (.nii, .nii.gz) — volumetric label maps or raw volumes
- **NRRD** (.nrrd) — volumetric data

**Output formats:**
- PLT (Tecplot) — thickness point cloud + projected mesh
- STL — 3D surface mesh
- **VTP (VTK PolyData)** — for **ParaView** interactive 3D visualization
- PNG report — static visualization with statistics

**Optimized with 3-phase acceleration:**
- **Phase 1**: Conjugate Gradient solver + RK4 integration + trilinear interpolation
- **Phase 2**: PyAMG algebraic multigrid preconditioner + Coupled PDE method (Wang et al. 2019)
- **Phase 3**: GPU-accelerated sparse solvers via CuPy + CUDA batch tracing

---

## 1. Setup Environment

In [ ]:
# Check GPU availability and type
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=gpu_name,memory.total,compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_info = result.stdout.strip()
print(f"GPU: {gpu_info}")

# Detect GPU tier for optimization guidance
if any(x in gpu_info.upper() for x in ['A100', 'H100', 'H200', 'L40']):
    print("\n✓ High-end datacenter GPU detected — full GPU acceleration available")
    print("  - TF32 tensor cores enabled (3x faster float32)")
    print("  - Large memory pool for full volume GPU processing")
elif any(x in gpu_info.upper() for x in ['V100', 'T4', 'L4', 'A10']):
    print("\n✓ Datacenter GPU detected — GPU acceleration available")
elif any(x in gpu_info.upper() for x in ['RTX', 'GTX']):
    print("\n✓ Consumer GPU detected — GPU acceleration available")
else:
    print("\n⚠ GPU type not recognized — will use CPU fallback if needed")

In [ ]:
# Install dependencies (quiet mode to reduce output)
!pip install -q numpy pydicom Pillow scipy pyamg numba

# Install CuPy matching CUDA version
import subprocess
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
nvcc_out = result.stdout
if 'release 12' in nvcc_out:
    !pip install -q cupy-cuda12x
elif 'release 11' in nvcc_out:
    !pip install -q cupy-cuda11x
else:
    print("CUDA version not detected, installing cupy-cuda12x by default")
    !pip install -q cupy-cuda12x

print("Dependencies installed:")
print("  - CuPy: GPU-accelerated sparse solvers + CUDA kernels")
print("  - PyAMG: Algebraic multigrid preconditioner")
print("  - Numba: JIT compilation fallback")

In [ ]:
# Verify CuPy GPU access + run benchmark
import time

try:
    import cupy as cp
    print(f"CuPy version: {cp.__version__}")

    cuda_ver = cp.cuda.runtime.runtimeGetVersion()
    print(f"CUDA runtime: {cuda_ver // 1000}.{(cuda_ver % 1000) // 10}")

    dev = cp.cuda.Device(0)
    cc = dev.compute_capability
    free, total = dev.mem_info
    print(f"GPU compute capability: {cc}")
    print(f"GPU memory: {free//1024//1024} MB free / {total//1024//1024} MB total")

    # Quick benchmark: sparse CG on GPU (the core operation)
    print("\nRunning GPU benchmark...")
    import cupyx.scipy.sparse as csp
    import cupyx.scipy.sparse.linalg as csla

    # Create test sparse system (3D Laplacian-like)
    N = 50000
    diag = cp.full(N, -6.0, dtype=cp.float64)
    off = cp.ones(N - 1, dtype=cp.float64)
    A_gpu = csp.diags([off, diag, off], [-1, 0, 1], shape=(N, N), format='csr')
    b_gpu = cp.random.randn(N).astype(cp.float64)

    # Warm up
    csla.cg(A_gpu, b_gpu, tol=1e-4, maxiter=50)
    cp.cuda.Stream.null.synchronize()

    # Benchmark
    t0 = time.time()
    for _ in range(5):
        x, info = csla.cg(A_gpu, b_gpu, tol=1e-4, maxiter=200)
    cp.cuda.Stream.null.synchronize()
    dt = (time.time() - t0) / 5

    print(f"  GPU sparse CG ({N} unknowns): {dt*1000:.1f} ms/solve")

    # Vectorized array ops benchmark
    vol = cp.random.randn(256, 256, 256).astype(cp.float32)
    t0 = time.time()
    for _ in range(10):
        grad = cp.abs(vol[2:] - vol[:-2]) * 0.5  # absolute gradient magnitude
    cp.cuda.Stream.null.synchronize()
    dt = (time.time() - t0) / 10
    print(f"  GPU gradient (256^3 volume): {dt*1000:.1f} ms")

    print("\nGPU acceleration ready")
    del vol, A_gpu, b_gpu, x

except Exception as e:
    print(f"CuPy not available: {e}")
    print("Pipeline will use CPU (NumPy) — slower but functional")

## 2. Clone Repository

In [ ]:
import os

# Clone the repository
REPO_URL = "https://github.com/sergiolaranjo/AutoAWT.git"  # Update with your repo URL
BRANCH = "claude/port-to-python-YUcq1"

if not os.path.exists("/content/AutoAWT"):
    !git clone -b {BRANCH} {REPO_URL} /content/AutoAWT
else:
    !cd /content/AutoAWT && git pull origin {BRANCH}

os.chdir("/content/AutoAWT/python")
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

## 3. Load Data

Choose one of the methods below:
- **Option A**: Upload BMP masks + DICOM (legacy pipeline)
- **Option B**: Mount Google Drive
- **Option C**: Use sample Phantom data from repository
- **Option D**: Direct DICOM volume import (new!)
- **Option E**: NIfTI / NRRD file import (new!)

In [ ]:
# ======= OPTION A: Upload files directly =======
# Uncomment and run this cell to upload data

# from google.colab import files
# import zipfile
#
# print("Upload your BMP masks as a ZIP file:")
# uploaded = files.upload()
# for fname in uploaded:
#     if fname.endswith('.zip'):
#         with zipfile.ZipFile(fname, 'r') as z:
#             z.extractall('/content/data/masks')
#         print(f"Extracted {fname} to /content/data/masks")
#
# print("\nUpload your DICOM files as a ZIP file:")
# uploaded = files.upload()
# for fname in uploaded:
#     if fname.endswith('.zip'):
#         with zipfile.ZipFile(fname, 'r') as z:
#             z.extractall('/content/data/dicom')
#         print(f"Extracted {fname} to /content/data/dicom")
#
# BMP_PATH = '/content/data/masks'
# DCM_PATH = '/content/data/dicom'

In [ ]:
# ======= OPTION B: Mount Google Drive =======
# Uncomment and run this cell if data is in Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
#
# # Set your data paths inside Google Drive
# BMP_PATH = '/content/drive/MyDrive/AutoAWT/masks'
# DCM_PATH = '/content/drive/MyDrive/AutoAWT/dicom'

In [ ]:
# ======= OPTION C: Use sample data from repo =======
BMP_PATH = '/content/AutoAWT/datasets/Phantom/masks_1'
DCM_PATH = '/content/AutoAWT/datasets/Phantom/ct'

# Verify data exists
import glob
bmp_files = sorted(glob.glob(os.path.join(BMP_PATH, '*.bmp')))
dcm_files = sorted(glob.glob(os.path.join(DCM_PATH, '*.dcm')))
print(f"BMP mask files: {len(bmp_files)}")
print(f"DICOM files: {len(dcm_files)}")

if len(bmp_files) == 0:
    print("WARNING: No BMP files found! Please check the path.")
if len(dcm_files) == 0:
    print("WARNING: No DICOM files found! Please check the path.")

In [ ]:
# ======= OPTION D: Direct DICOM volume import =======
# Reads pixel data directly from DICOM files and creates binary mask
# No external segmentation needed — uses HU threshold
# Uncomment to use:

# from google.colab import files
# import zipfile
#
# print("Upload your DICOM series as a ZIP file:")
# uploaded = files.upload()
# for fname in uploaded:
#     if fname.endswith('.zip'):
#         with zipfile.ZipFile(fname, 'r') as z:
#             z.extractall('/content/data/dicom_volume')
#
# DICOM_VOLUME_PATH = '/content/data/dicom_volume'
# HU_THRESHOLD = None  # Set to None for auto (Otsu), or e.g. -200 for manual
# INPUT_MODE = 'dicom_direct'

## 4. Run AutoAWT Pipeline

In [ ]:
# ======= OPTION E: NIfTI / NRRD import =======
# For pre-segmented volumes or raw data in standard formats
# Uncomment to use:

# !pip install nibabel  # For NIfTI support
# # or: !pip install pynrrd  # For NRRD support
#
# from google.colab import files
# print("Upload your NIfTI or NRRD file:")
# uploaded = files.upload()
# NIFTI_PATH = list(uploaded.keys())[0]
# HU_THRESHOLD = 0  # For label maps use 0; for raw volumes use Otsu (None)
# INPUT_MODE = 'nifti'

In [ ]:
import os
import sys
import glob
import time
import numpy as np

# Add python dir to path
sys.path.insert(0, '/content/AutoAWT/python')

from backend import xp, GPU_AVAILABLE, to_numpy, to_device, gpu_info
from utils import get_file_list, read_bmp_files, read_dcm_for_pixel_spacing
from utils import export_bmp, compute_fill_space, arr_logical_and
from qhull import QHull, CTview
from wt import WT
from marching_cubes import MarchingCube

# Show backend info
info = gpu_info()
if info['available']:
    print(f"Backend: GPU — {info['name']}")
    print(f"  Memory: {info.get('gpu_free_mb', '?')} MB free / {info.get('gpu_total_mb', '?')} MB total")
    print(f"  Compute capability: {info['compute_capability']}")
else:
    print("Backend: CPU (NumPy)")

# Load data based on selected input mode
INPUT_MODE = locals().get('INPUT_MODE', 'bmp')

if INPUT_MODE == 'dicom_direct':
    from io_formats import load_volume
    DICOM_VOLUME_PATH = locals().get('DICOM_VOLUME_PATH', '/content/data/dicom_volume')
    HU_THRESHOLD = locals().get('HU_THRESHOLD', None)
    g_wall_mask, volume_size, g_pixel_spacing, volume_position, patient_id = \
        load_volume(DICOM_VOLUME_PATH, fmt='dicom', threshold=HU_THRESHOLD)
    DATA_PATH = DICOM_VOLUME_PATH

elif INPUT_MODE in ('nifti', 'nrrd'):
    from io_formats import load_volume
    NIFTI_PATH = locals().get('NIFTI_PATH', '')
    HU_THRESHOLD = locals().get('HU_THRESHOLD', None)
    if not NIFTI_PATH:
        raise ValueError("NIFTI_PATH not set. Please set it in the Option E cell.")
    g_wall_mask, volume_size, g_pixel_spacing, volume_position, patient_id = \
        load_volume(NIFTI_PATH, threshold=HU_THRESHOLD)
    DATA_PATH = os.path.dirname(NIFTI_PATH) or '.'

else:
    # Legacy BMP + DICOM mode (Options A/B/C)
    file_names = get_file_list(BMP_PATH, '.bmp')
    print(f"\nFound {len(file_names)} BMP files")
    g_wall_mask, volume_size = read_bmp_files(file_names)
    if g_wall_mask is None:
        raise RuntimeError("Failed to read BMP files. Check that BMP_PATH is correct.")

    dcm_files = get_file_list(DCM_PATH, '.dcm')
    g_pixel_spacing, volume_position, patient_id = read_dcm_for_pixel_spacing(dcm_files)
    DATA_PATH = BMP_PATH

w, h, d = volume_size[0], volume_size[1], volume_size[2]
print(f"\nVolume: {w} x {h} x {d}")
print(f"Voxel spacing: {g_pixel_spacing}")
print(f"Patient ID: {patient_id}")
print(f"Non-zero voxels: {np.count_nonzero(g_wall_mask):,}")

In [ ]:
# Step 2: Convex Hull Generation
t0 = time.time()

merged_hull = np.zeros((d, h, w), dtype=np.uint16)
hull_mask = np.zeros((d, h, w), dtype=np.uint16)

# Axial convex hulls
print("Computing axial convex hulls...")
for i in range(d):
    qh = QHull()
    qh.set_point_cloud(g_wall_mask, volume_size, i, CTview.AXIAL)
    if len(qh.get_point_cloud()) < 1:
        continue
    qh.initialize()
    convex_points = qh.get_drawable_points()
    compute_fill_space(merged_hull, convex_points, i, (w, h, d), CTview.AXIAL)

# Sagittal convex hulls
print("Computing sagittal convex hulls...")
for i in range(w):
    qh = QHull()
    qh.set_point_cloud(g_wall_mask, volume_size, i, CTview.SAGITTAL)
    if len(qh.get_point_cloud()) < 1:
        continue
    qh.initialize()
    convex_points = qh.get_drawable_points()
    compute_fill_space(hull_mask, convex_points, i, (w, h, d), CTview.SAGITTAL)

arr_logical_and(merged_hull, hull_mask)

# Coronal convex hulls
hull_mask[:] = 0
print("Computing coronal convex hulls...")
for i in range(h):
    qh = QHull()
    qh.set_point_cloud(g_wall_mask, volume_size, i, CTview.CORONAL)
    if len(qh.get_point_cloud()) < 1:
        continue
    qh.initialize()
    convex_points = qh.get_drawable_points()
    compute_fill_space(hull_mask, convex_points, i, (w, h, d), CTview.CORONAL)

arr_logical_and(merged_hull, hull_mask)
del hull_mask

print(f"Convex hull voxels: {np.count_nonzero(merged_hull)}")
print(f"Time: {time.time()-t0:.1f}s")

In [ ]:
# Step 3: Wall Thickness Computation
t0 = time.time()

results_path = os.path.join(DATA_PATH, "Results")
os.makedirs(results_path, exist_ok=True)

wt_algorithms = WT(
    results_path, volume_size, g_pixel_spacing,
    volume_position, g_wall_mask, merged_hull
)

print("Detecting epi/endo...")
wt_algorithms.detect_epi_endo(None)
t1 = time.time()
print(f"Epi-endo detection: {t1-t0:.1f}s")

print("\nComputing wall thickness...")
wt_algorithms.eval_wt()
t2 = time.time()
print(f"Wall thickness computation: {t2-t1:.1f}s")

# Copy results
g_wall_mask[:] = wt_algorithms.get_chamber_mask()
endo_vertices_list = list(wt_algorithms.m_endo_vertices_list)
print(f"\nEndo vertices: {len(endo_vertices_list)}")
print(f"Total WT time: {time.time()-t0:.1f}s")

del wt_algorithms

In [ ]:
# Step 4: Surface Mesh Generation (Marching Cubes)
t0 = time.time()

print("Generating surface mesh...")
surfaces = MarchingCube(
    results_path, g_wall_mask, volume_size,
    g_pixel_spacing, volume_position, MarchingCube.FROM_HOST
)
surfaces.compute_isosurface(g_wall_mask, MarchingCube.FROM_HOST)
surfaces.save_mesh_info(
    os.path.join(results_path, f"WT(projected)-{patient_id}"),
    endo_vertices_list
)

if 'merged_hull' in dir():
    del merged_hull
print(f"Surface mesh generation: {time.time()-t0:.1f}s")
print("\nAutoAWT processing complete!")

In [ ]:
# Step 5: Export for ParaView (VTP format)
import os
from io_formats import plt_to_vtp, stl_to_vtp

print("Exporting VTP files for ParaView...")

endo_plt = os.path.join(results_path, "WT-endo.plt")
if os.path.exists(endo_plt):
    vtp = plt_to_vtp(endo_plt)
    print(f"  Created: {vtp}")

# Projected thickness mesh
for f in os.listdir(results_path):
    if f.startswith("WT(projected)") and f.endswith(".plt"):
        vtp = plt_to_vtp(os.path.join(results_path, f))
        print(f"  Created: {vtp}")
        break

# Surface mesh with thickness mapping
stl_file = os.path.join(results_path, "surface_mesh.stl")
if os.path.exists(stl_file) and os.path.exists(endo_plt):
    vtp = stl_to_vtp(stl_file, thickness_plt=endo_plt)
    print(f"  Created: {vtp}")

print("\nVTP files ready for ParaView!")
print("Download and open with: paraview <file>.vtp")

## 5. Results

In [ ]:
# List output files
print(f"Results directory: {results_path}")
print("\nOutput files:")
for root, dirs, ffiles in os.walk(results_path):
    for f in sorted(ffiles):
        fpath = os.path.join(root, f)
        fsize = os.path.getsize(fpath)
        rel = os.path.relpath(fpath, results_path)
        print(f"  {rel}: {fsize/1024:.1f} KB")

In [ ]:
# Visualize wall thickness from PLT file
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt_file = os.path.join(results_path, "WT-endo.plt")
if os.path.exists(plt_file):
    # Parse PLT file
    data = []
    with open(plt_file, 'r') as f:
        for line in f:
            if line.startswith('VARIABLES') or line.startswith('ZONE'):
                continue
            parts = line.strip().split()
            if len(parts) == 4:
                try:
                    data.append([float(x) for x in parts])
                except ValueError:
                    continue

    data = np.array(data)
    if len(data) > 0:
        x, y, z, wt = data[:, 0], data[:, 1], data[:, 2], data[:, 3]

        print(f"Wall thickness statistics:")
        print(f"  Mean: {np.mean(wt):.3f} mm")
        print(f"  Std:  {np.std(wt):.3f} mm")
        print(f"  Min:  {np.min(wt):.3f} mm")
        print(f"  Max:  {np.max(wt):.3f} mm")
        print(f"  Points: {len(data)}")

        # 3D scatter plot with thickness coloring
        fig = plt.figure(figsize=(14, 6))

        ax1 = fig.add_subplot(121, projection='3d')
        sc = ax1.scatter(x, y, z, c=wt, cmap='jet', s=1, alpha=0.6)
        ax1.set_xlabel('X (mm)')
        ax1.set_ylabel('Y (mm)')
        ax1.set_zlabel('Z (mm)')
        ax1.set_title('Endocardial Wall Thickness')
        plt.colorbar(sc, ax=ax1, label='Thickness (mm)', shrink=0.6)

        # Histogram
        ax2 = fig.add_subplot(122)
        ax2.hist(wt[wt > 0], bins=50, color='steelblue', edgecolor='black', alpha=0.8)
        ax2.set_xlabel('Wall Thickness (mm)')
        ax2.set_ylabel('Count')
        ax2.set_title('Wall Thickness Distribution')
        ax2.axvline(np.mean(wt[wt > 0]), color='red', linestyle='--', label=f'Mean: {np.mean(wt[wt > 0]):.2f} mm')
        ax2.legend()

        plt.tight_layout()
        plt.show()
    else:
        print("No data points found in PLT file")
else:
    print(f"PLT file not found at: {plt_file}")

In [ ]:
# Visualize Epi-Endo labels (middle slices)
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

epi_endo_path = os.path.join(results_path, "Epi-Endo")
if os.path.exists(epi_endo_path):
    bmp_files_ee = sorted(glob.glob(os.path.join(epi_endo_path, '*.bmp')))
    n_slices = len(bmp_files_ee)
    print(f"Epi-Endo slices: {n_slices}")

    if n_slices > 0:
        # Show 6 evenly spaced slices
        indices = np.linspace(0, n_slices - 1, min(6, n_slices), dtype=int)
        fig, axes = plt.subplots(1, len(indices), figsize=(18, 4))
        if len(indices) == 1:
            axes = [axes]
        for ax, idx in zip(axes, indices):
            img = np.array(Image.open(bmp_files_ee[idx]))
            ax.imshow(img, cmap='viridis')
            ax.set_title(f'Slice {idx}')
            ax.axis('off')
        plt.suptitle('Epi-Endo Labels (1=endo, 2=wall, 3=epi)', y=1.02)
        plt.tight_layout()
        plt.show()
else:
    print("Epi-Endo directory not found")

## 6. Download Results

In [ ]:
# Create ZIP archive of results
import shutil

zip_path = '/content/AutoAWT_Results'
shutil.make_archive(zip_path, 'zip', results_path)
print(f"Results archived: {zip_path}.zip")
print(f"Size: {os.path.getsize(zip_path + '.zip') / (1024*1024):.1f} MB")

# Download (uncomment to download)
# from google.colab import files
# files.download(zip_path + '.zip')